# 03_dim_artists

DML: dim_artists — Artist dimension, PK: artist_id.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

src = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_artists")
    .filter(F.col("ingestion_date") == ingestion_date)
    .filter(F.col("run_id") == run_id)
    .select("artist_id", "artist_name", "genres", "popularity", "followers_total", "ingestion_date")
    .withColumn("primary_genre", F.element_at("genres", 1))
    .withColumn("_valid_from", F.col("ingestion_date"))
    .drop("ingestion_date")
    .dropDuplicates(["artist_id"])
)

upsert_delta(src, f"{CATALOG}.{SILVER_SCHEMA}.dim_artists", ["artist_id"])
print(f"dim_artists: {src.count()} rows upserted")